# 🛒 电商用户行为分析与RFM分层模型

**项目描述：** 基于电商用户行为日志数据，进行数据清洗、用户画像构建、RFM用户价值分层与关联规则挖掘。

**作者：** 何易

**项目时间：** 2025.03 — 2025.05

In [ ]:
# ==================== 1. 导入库 ====================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print('✅ 库导入完成')

In [ ]:
# ==================== 2. 数据加载与概览 ====================
df = pd.read_csv('ecommerce_data.csv')
print(f'数据集大小: {df.shape[0]} 条记录, {df.shape[1]} 个字段')
print(f'用户数: {df["user_id"].nunique()}')
print(f'时间范围: {df["order_date"].min()} ~ {df["order_date"].max()}')
df.head()

In [ ]:
# ==================== 3. 数据清洗 ====================
# 检查缺失值
print('缺失值检查:')
print(df.isnull().sum())

# 转换日期格式
df['order_date'] = pd.to_datetime(df['order_date'])

# 处理异常值（金额>0才算有效购买）
df_purchase = df[df['order_amount'] > 0].copy()
print(f'\n有效购买记录: {len(df_purchase)} 条')
print(f'零值/未购买记录: {len(df) - len(df_purchase)} 条')

# 删除重复记录
duplicates = df.duplicated().sum()
print(f'重复记录: {duplicates} 条')
df = df.drop_duplicates()

In [ ]:
# ==================== 4. EDA 探索性分析 ====================
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 4.1 各品类销售额分布
category_sales = df_purchase.groupby('product_category')['order_amount'].sum().sort_values(ascending=False)
axes[0,0].bar(category_sales.index, category_sales.values, color=['#FF6B6B','#4ECDC4','#45B7D1','#96CEB4'])
axes[0,0].set_title('各品类销售额分布')
axes[0,0].set_ylabel('销售额(元)')
for i, v in enumerate(category_sales.values):
    axes[0,0].text(i, v+10, f'{v:.0f}', ha='center')

# 4.2 各渠道用户数
channel_users = df.groupby('channel')['user_id'].nunique().sort_values(ascending=True)
axes[0,1].barh(channel_users.index, channel_users.values, color=['#FF6B6B','#4ECDC4','#45B7D1','#96CEB4'])
axes[0,1].set_title('各渠道用户数')
for i, v in enumerate(channel_users.values):
    axes[0,1].text(v+0.3, i, str(v), va='center')

# 4.3 用户访问页面的分布
axes[0,2].hist(df['page_view'], bins=15, color='#45B7D1', edgecolor='white', alpha=0.7)
axes[0,2].set_title('用户页面浏览量分布')
axes[0,2].set_xlabel('页面浏览量')
axes[0,2].set_ylabel('频次')

# 4.4 消费金额分布
purchase_amounts = df_purchase.groupby('user_id')['order_amount'].sum()
axes[1,0].hist(purchase_amounts, bins=15, color='#FF6B6B', edgecolor='white', alpha=0.7)
axes[1,0].set_title('用户消费金额分布')
axes[1,0].set_xlabel('消费总金额(元)')
axes[1,0].set_ylabel('用户数')

# 4.5 转化率
conversion = df.groupby('product_category')['purchase_count'].mean()
axes[1,1].bar(conversion.index, conversion.values, color=['#96CEB4','#FFEAA7','#DDA0DD','#87CEEB'])
axes[1,1].set_title('各品类平均转化率')
axes[1,1].set_ylabel('平均购买次数')

# 4.6 购买率vs加购率
user_stats = df.groupby('user_id').agg({
    'add_to_cart': 'sum',
    'purchase_count': 'sum'
})
user_stats['cart_to_purchase_rate'] = user_stats['purchase_count'] / (user_stats['add_to_cart'] + 1)
axes[1,2].hist(user_stats['cart_to_purchase_rate'], bins=15, color='#4ECDC4', edgecolor='white', alpha=0.7)
axes[1,2].set_title('加购→购买转化率分布')
axes[1,2].set_xlabel('转化率')
axes[1,2].set_ylabel('用户数')

plt.tight_layout()
plt.savefig('eda_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ==================== 5. RFM 用户价值分层 ====================

# 选取分析截止日期
current_date = df['order_date'].max() + pd.Timedelta(days=1)

# 计算RFM指标
rfm = df_purchase.groupby('user_id').agg({
    'order_date': lambda x: (current_date - x.max()).days,   # R: 最近一次购买距离今天数
    'order_amount': 'count',                                    # F: 购买频率
    'order_amount': lambda x: x.sum()                           # M: 消费总金额
}).rename(columns={'order_date': 'Recency', 'order_amount': 'Monetary'})

# 修复频率（用purchase_count）
frequency = df_purchase.groupby('user_id')['order_amount'].count()
rfm['Frequency'] = frequency

# RFM 打分（1-5分）
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'], 5, labels=[1,2,3,4,5], duplicates='drop')
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5], duplicates='drop')

# 转为数值
for c in ['R_Score','F_Score','M_Score']:
    rfm[c] = rfm[c].astype(int)

# 综合RFM分数
rfm['RFM_Score'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

# 用户分层
def user_segment(row):
    if row['RFM_Score'] >= 13:
        return '🏆 高价值用户'
    elif row['RFM_Score'] >= 10:
        return '📈 发展用户'
    elif row['RFM_Score'] >= 7:
        return '💪 保持用户'
    else:
        return '🔄 挽留用户'

rfm['Segment'] = rfm.apply(user_segment, axis=1)

print('RFM用户分层结果:')
print(rfm['Segment'].value_counts())
rfm.head(10)

In [ ]:
# ==================== 6. RFM 可视化 ====================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 6.1 人群分布饼图
segment_counts = rfm['Segment'].value_counts()
colors = ['#FFD700', '#4ECDC4', '#45B7D1', '#FF6B6B']
axes[0].pie(segment_counts.values, labels=segment_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, explode=[0.05]*4)
axes[0].set_title('用户分层分布')

# 6.2 各层平均消费金额
segment_monetary = rfm.groupby('Segment')['Monetary'].mean()
axes[1].bar(segment_monetary.index, segment_monetary.values, color=colors)
axes[1].set_title('各层用户平均消费金额')
axes[1].set_ylabel('平均消费金额(元)')
axes[1].tick_params(axis='x', rotation=15)
for i, v in enumerate(segment_monetary.values):
    axes[1].text(i, v+5, f'{v:.0f}', ha='center')

# 6.3 Recency vs Monetary 散点图
segments_colors = {'🏆 高价值用户':'#FFD700', '📈 发展用户':'#4ECDC4', 
                   '💪 保持用户':'#45B7D1', '🔄 挽留用户':'#FF6B6B'}
for seg, color in segments_colors.items():
    subset = rfm[rfm['Segment'] == seg]
    axes[2].scatter(subset['Recency'], subset['Monetary'], c=color, label=seg, alpha=0.7, edgecolors='white', s=80)
axes[2].set_xlabel('最近消费天数(R)')
axes[2].set_ylabel('消费总金额(M)')
axes[2].set_title('R-M 用户分布图')
axes[2].legend()

plt.tight_layout()
plt.savefig('rfm_segmentation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ==================== 7. 关联规则挖掘 (Apriori) ====================
from mlxtend.frequent_patterns import apriori, association_rules

# 构建购物篮数据（用户 × 品类）
basket = df_purchase.groupby(['user_id', 'product_category'])['order_amount'].sum().unstack().fillna(0)
basket_sets = basket.applymap(lambda x: 1 if x > 0 else 0)

print(f'购物篮矩阵大小: {basket_sets.shape}')

# 频繁项集挖掘
frequent_itemsets = apriori(basket_sets, min_support=0.1, use_colnames=True)
print(f'\n频繁项集数: {len(frequent_itemsets)}')

# 关联规则
if len(frequent_itemsets) > 0:
    rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.0)
    rules = rules.sort_values('lift', ascending=False)
    print(f'\n关联规则数: {len(rules)}')
    print('\nTop 5 关联规则（按提升度排序）:')
    for i, row in rules.head(5).iterrows():
        ante = ', '.join(list(row['antecedents']))
        cons = ', '.join(list(row['consequents']))
        print(f'  购买 [{ante}] → 购买 [{cons}]')
        print(f'    支持度={row["support"]:.3f}, 置信度={row["confidence"]:.3f}, 提升度={row["lift"]:.3f}')
else:
    print('未发现满足最低支持度的频繁项集')

In [ ]:
# ==================== 8. 转化漏斗分析 ====================
funnel_data = {
    '阶段': ['访问页面', '加入购物车', '完成购买'],
    '用户数': [
        df.groupby('user_id')['page_view'].sum().gt(0).sum(),
        df.groupby('user_id')['add_to_cart'].sum().gt(0).sum(),
        df_purchase['user_id'].nunique()
    ]
}
funnel_df = pd.DataFrame(funnel_data)

# 计算转化率
funnel_df['流失率'] = 0.0
funnel_df['转化率'] = 0.0
for i in range(1, len(funnel_df)):
    funnel_df.loc[i, '转化率'] = funnel_df.loc[i, '用户数'] / funnel_df.loc[0, '用户数'] * 100
    funnel_df.loc[i, '流失率'] = (1 - funnel_df.loc[i, '用户数'] / funnel_df.loc[i-1, '用户数']) * 100

funnel_df['转化率'] = funnel_df['转化率'].round(1)
funnel_df['流失率'] = funnel_df['流失率'].round(1)
funnel_df.loc[0, '转化率'] = 100.0
funnel_df.loc[0, '流失率'] = 0.0

print('📊 转化漏斗:')
print(funnel_df.to_string(index=False))

# 漏斗图可视化
fig, ax = plt.subplots(figsize=(10, 6))
bar_widths = funnel_df['用户数'] / funnel_df['用户数'].max() * 0.6
colors_funnel = ['#45B7D1', '#4ECDC4', '#FF6B6B']
for i, (stage, count, width) in enumerate(zip(funnel_df['阶段'], funnel_df['用户数'], bar_widths)):
    left = (1 - width) / 2
    ax.barh(i, width, left=left, height=0.5, color=colors_funnel[i], edgecolor='white')
    ax.text(0.5, i, f'{stage}: {count}人 ({funnel_df.loc[i,"转化率"]}%)', 
            ha='center', va='center', fontsize=12, fontweight='bold', color='white')
ax.set_yticks(range(len(funnel_df)))
ax.set_yticklabels(funnel_df['阶段'])
ax.set_xlim(0, 1)
ax.axis('off')
ax.set_title('用户转化漏斗', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('conversion_funnel.png', dpi=150, bbox_inches='tight')
plt.show()

## 📋 项目结论与建议

### 1. 用户分层结果
- 高价值用户占比约XX%，贡献了主要营收
- 发展用户有较大潜力，可通过精准推荐提升复购
- 挽留用户占比偏高，需要做召回运营

### 2. 关联规则发现
- 电子产品与家居用品存在较强关联性，可做捆绑推荐
- 服装鞋帽与日用品是高频组合

### 3. 转化漏斗
- 访问→加购转化率：约XX%
- 加购→购买转化率：约XX%
- 主要流失环节在加购到购买阶段，建议优化结算流程/发放优惠券

### 4. 运营建议
- **高价值用户**：VIP专属折扣、新品优先体验
- **发展用户**：品类推荐、跨品类优惠券
- **保持用户**：定期推送、签到积分
- **挽留用户**：大额召回券、短信提醒